<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-1 · Part 2: Unconstrained, Bound-Constrained, and Generally Constrained Optimization

**The same cooling decision can be selectable in one formulation and rejected in another because the requirements define eligibility.**

Part 1 introduced \(x\), \(y=\operatorname{Sim}(x)\), \(f\), \(g\), and \(h\). This part keeps the classroom decision, simulator, and objective fixed. Only the constraints change.

| Kept fixed | Changed here | Resulting classification |
|:---|:---|:---|
| $x$, $y=\operatorname{Sim}(x)$, $f(y)$ | Bounds, inequalities, and equalities | Unconstrained, bound-constrained, or generally constrained |


### 1 · Write every requirement as a residual

An inequality residual uses one sign convention:

> $\displaystyle g_j(x,y)\le0.$

For example, the energy requirement \(E(u)\le E_{\max}\) becomes \(g_E(x,y)=E(u)-E_{\max}\le0\). The sign of the residual gives its status:

| Residual value | Interpretation |
|:---:|:---|
| $g_j<0$ | Satisfied with slack |
| $g_j=0$ | Active and satisfied |
| $g_j>0$ | Violated |

The classroom requirements become:

| Requirement | Residual form |
|:---|:---|
| $u_{\min}\le u_{\mathrm{early}}\le u_{\max}$ | $u_{\min}-u_{\mathrm{early}}\le0$, $u_{\mathrm{early}}-u_{\max}\le0$ |
| $u_{\min}\le u_{\mathrm{late}}\le u_{\max}$ | $u_{\min}-u_{\mathrm{late}}\le0$, $u_{\mathrm{late}}-u_{\max}\le0$ |
| $T_{\min}\le T_t\le T_{\max}$ | $T_{\min}-T_t\le0$, $T_t-T_{\max}\le0$ |
| $E(u)\le E_{\max}$ | $E(u)-E_{\max}\le0$ |

The fixed limits are \(u_{\min}=0\), \(u_{\max}=5\), \(T_{\min}=20\,^\circ\mathrm C\), \(T_{\max}=30\,^\circ\mathrm C\), and \(E_{\max}=60\).

The next figure compares one feasible and one infeasible classroom candidate using the same residual definitions.


In [ ]:
import numpy as np


TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Decision domain and requirement limits
MIN_COOLING = 0.0
MAX_COOLING = 5.0
MIN_TEMPERATURE = 20.0
MAX_TEMPERATURE = 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    """Expand x=[early cooling, late cooling] into the 12-step schedule."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[
        np.full(6, early_cooling),
        np.full(6, late_cooling),
    ]


def simulation_model(x):
    """Return y=Sim(x): the state path and raw performance outputs."""
    cooling_schedule = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]

    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )

    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule**2)
    return {
        "temperatures": temperatures,
        "discomfort": float(discomfort),
        "energy": float(energy),
    }


def objective_function(y, energy_weight=1.0):
    """Return f(y; lambda_E)=D(u)+lambda_E E(u)."""
    return y["discomfort"] + float(energy_weight) * y["energy"]


def inequality_constraints(x, y, energy_limit=MAX_ENERGY):
    """Return residuals in the feasible form g_j(x,y) <= 0."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    temperatures = y["temperatures"][1:]
    return {
        "early lower": MIN_COOLING - early_cooling,
        "early upper": early_cooling - MAX_COOLING,
        "late lower": MIN_COOLING - late_cooling,
        "late upper": late_cooling - MAX_COOLING,
        "temperature lower": MIN_TEMPERATURE - temperatures.min(),
        "temperature upper": temperatures.max() - MAX_TEMPERATURE,
        "energy": y["energy"] - float(energy_limit),
    }


def equality_constraints(x, y):
    """Return state-equation residuals h_t(x,y), which should equal zero."""
    cooling_schedule = expand_decision(x)
    temperatures = y["temperatures"]
    residuals = []
    for step, (outdoor, people, cooling) in enumerate(
        zip(OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule)
    ):
        predicted_next = (
            temperatures[step]
            + WEATHER_EXCHANGE * (outdoor - temperatures[step])
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
        residuals.append(temperatures[step + 1] - predicted_next)
    return np.asarray(residuals)


def evaluate_formulation(x, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate x, y, f, g, and h for one candidate decision."""
    x = tuple(map(float, x))
    y = simulation_model(x)
    g = inequality_constraints(x, y, energy_limit)
    h = equality_constraints(x, y)
    feasible = all(value <= 1e-10 for value in g.values()) and np.allclose(h, 0.0)
    return {
        "x": x,
        "y": y,
        "f": objective_function(y, energy_weight),
        "g": g,
        "h": h,
        "energy_weight": float(energy_weight),
        "energy_limit": float(energy_limit),
        "feasible": bool(feasible),
    }


def enumerate_candidates(step=0.5, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate a stated finite candidate grid."""
    levels = np.arange(MIN_COOLING, MAX_COOLING + step / 2, step)
    return [
        evaluate_formulation((early, late), energy_weight, energy_limit)
        for early in levels
        for late in levels
    ]


def select_best(records, key="f"):
    """Select the lowest-valued feasible record for the requested key."""
    return min(
        (record for record in records if record["feasible"]),
        key=lambda record: record[key] if key != "discomfort" else record["y"][key],
    )


def pareto_front(records):
    """Return nondominated feasible records for minimizing discomfort and energy."""
    feasible_records = [record for record in records if record["feasible"]]
    nondominated = []
    for candidate in feasible_records:
        candidate_d = candidate["y"]["discomfort"]
        candidate_e = candidate["y"]["energy"]
        dominated = any(
            other["y"]["discomfort"] <= candidate_d
            and other["y"]["energy"] <= candidate_e
            and (
                other["y"]["discomfort"] < candidate_d
                or other["y"]["energy"] < candidate_e
            )
            for other in feasible_records
        )
        if not dominated:
            nondominated.append(candidate)
    return sorted(nondominated, key=lambda record: record["y"]["energy"])

import sys
from types import SimpleNamespace

import matplotlib

def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt

def show_constraint_residuals(evaluate_formulation):
    """Print and plot residuals for one feasible and one infeasible candidate."""
    plt = _pyplot()
    records = [evaluate_formulation((3.0, 2.0)), evaluate_formulation((4.0, 4.0))]
    print("Constraint residuals: every g must be <= 0")
    header = f"{'constraint':>20} | {'x=[3,2]^T':>10} | {'x=[4,4]^T':>10}"
    print(header)
    print("-" * len(header))
    for name in records[0]["g"]:
        print(
            f"{name:>20} | {records[0]['g'][name]:>10.2f} | "
            f"{records[1]['g'][name]:>10.2f}"
        )
    print(
        "max |h_t|: "
        f"{np.abs(records[0]['h']).max():.2e} and "
        f"{np.abs(records[1]['h']).max():.2e}"
    )

    figure, axes = plt.subplots(1, 2, figsize=(10.2, 4.6), sharey=True)
    names = list(records[0]["g"])
    display_names = [name.replace(" ", "\n") for name in names]
    for axis, record in zip(axes, records):
        values = np.array([record["g"][name] for name in names])
        colors = ["#c92a2a" if value > 0 else "#087f5b" for value in values]
        axis.barh(np.arange(len(values)), values, color=colors, alpha=0.82)
        axis.axvline(0, color="black", linewidth=1.2)
        axis.set_xscale("symlog", linthresh=0.1)
        axis.set(
            yticks=np.arange(len(values)),
            yticklabels=display_names,
            xlabel="Constraint residual $g_j$ (symlog scale)",
            title=(
                rf"$x=[{record['x'][0]:g},{record['x'][1]:g}]^{{\mathsf{{T}}}}$ · "
                f"{'feasible' if record['feasible'] else 'infeasible'}"
            ),
        )
        axis.grid(axis="x", alpha=0.25)
    figure.suptitle("Negative is allowed · positive is a violation", fontsize=12)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
    plt.close(figure)
    return records

residual_examples = show_constraint_residuals(evaluate_formulation)


For \(x=[3,2]^{\mathsf{T}}\), every bar ends at or left of zero, so the candidate is feasible. For \(x=[4,4]^{\mathsf{T}}\), the lower-temperature and energy residuals are positive, so the candidate is infeasible. One violation is enough for rejection.

Equality residuals use \(h_k(x,y)=0\). If the simulator produces the state path by repeatedly applying \(F\), the state equation is already enforced inside \(y=\operatorname{Sim}(x)\). If the states are independent optimization variables, the dynamics must instead appear explicitly as \(h_t=0\).


### 2 · The restriction set determines the constraint class

All three formulations below may use the same simulator and objective. Their acceptable decision sets differ.

| Formulation | Restrictions | Meaning |
|:---|:---|:---|
| Unconstrained | $x\in\mathbb R^d$ | No explicit requirement restricts $x$ |
| Bound-constrained | $\ell\le x\le r$ | Only lower and upper decision bounds restrict $x$ |
| Generally constrained | $g_j(x,y)\le0$ or $h_k(x,y)=0$ | General inequalities or equalities restrict eligibility |

Here, \(d\) is the number of decision components. The vectors \(\ell\) and \(r\) contain lower and upper bounds. A bounded domain such as \(x\in[0,5]^2\) is therefore not unconstrained.

The classroom formulation is generally constrained because temperature and energy residuals restrict the decision in addition to its bounds.

The controlled comparison below uses the same candidate \(x=[4,4]^{\mathsf{T}}\), the same response \(y\), and the same score \(f\). Only the restriction set changes.


In [ ]:
candidate = evaluate_formulation((4.0, 4.0))
early, late = candidate["x"]
inside_bounds = 0.0 <= early <= 5.0 and 0.0 <= late <= 5.0

print(f"same response: D={candidate['y']['discomfort']:.2f}, "
      f"E={candidate['y']['energy']:.2f}, f={candidate['f']:.2f}")
print(f"unconstrained eligibility: True")
print(f"bound-constrained eligibility: {inside_bounds}")
print(f"generally constrained eligibility: {candidate['feasible']}")


### 3 · Changing constraints changes the problem, not the candidate

The candidate \(x=[4,4]^{\mathsf{T}}\) has the same temperature path, \(D\), \(E\), and \(f\) in all three cases. It is allowed by the unconstrained and bound-constrained formulations but rejected by the classroom formulation.

Removing a requirement creates a different optimization formulation. It is not merely a different search algorithm. Conversely, changing from grid search to another algorithm does not remove or add a constraint.

Simple bounds may be represented inside \(\mathcal X\) or as residuals. A solver implementation should include each bound once, not duplicate it in both places.


### Takeaway

Classify the constraint structure by asking what restricts eligibility:

> **no explicit restriction → unconstrained · bounds only → bound-constrained · general \(g\) or \(h\) → generally constrained**

Always evaluate one candidate in this order: compute the response, calculate every residual, reject the candidate if any inequality residual is positive or any equality residual is nonzero, and only then compare objective values.

Part 3 keeps the constraints fixed and changes the decision domain \(\mathcal X\).
